### APEX WEALTH DATA PIPELINE


In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [3]:

load_dotenv()
api_key = os.getenv('API_KEY')

In [4]:
API_KEY = os.getenv('API_KEY')

In [5]:
api_key

'0fb28af206c64e9da726186651ad43c2'

In [6]:
symbol = "AAPL"
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"


In [7]:
#Get response from the API
response = requests.get(url)
response.status_code

200

In [8]:
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "apikey": API_KEY
}


In [9]:


resp = requests.get(url, params=params)
print("Status code:", resp.status_code)  # Should be 200

data = resp.json()
print("Keys in JSON:", data.keys())      # Should include 'values'


Status code: 200
Keys in JSON: dict_keys(['meta', 'values', 'status'])


In [10]:
#get data in json format
data = response.json()
data


{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-02-04 15:37:00',
   'open': '277.67',
   'high': '277.785',
   'low': '277.57',
   'close': '277.7',
   'volume': '9808'},
  {'datetime': '2026-02-04 15:36:00',
   'open': '277.33',
   'high': '277.74',
   'low': '277.33',
   'close': '277.65',
   'volume': '11560'},
  {'datetime': '2026-02-04 15:35:00',
   'open': '277.28',
   'high': '277.51',
   'low': '277',
   'close': '277.34',
   'volume': '12067'},
  {'datetime': '2026-02-04 15:34:00',
   'open': '277.45',
   'high': '277.62',
   'low': '277.22',
   'close': '277.295',
   'volume': '11168'},
  {'datetime': '2026-02-04 15:33:00',
   'open': '277.345',
   'high': '277.555',
   'low': '277.34',
   'close': '277.46',
   'volume': '3324'},
  {'datetime': '2026-02-04 15:32:00',
   'open': '277.06',
   'high': '277.5',
 

In [11]:
df = pd.DataFrame(data["values"])
df.head()

,datetime,open,high,low,close,volume
0,2026-02-04 15:37:00,277.67,277.785,277.57,277.7,9808
1,2026-02-04 15:36:00,277.33,277.74,277.33,277.65,11560
2,2026-02-04 15:35:00,277.28,277.51,277,277.34,12067
3,2026-02-04 15:34:00,277.45,277.62,277.22,277.295,11168
4,2026-02-04 15:33:00,277.345,277.555,277.34,277.46,3324


In [12]:
#transform data to dataframe
def transform_data(data):
    #extract values
    time_series = data['values']

    #convert to dataframe
    df = pd.DataFrame(time_series)

    #convert to proper datatypes
    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int' })
    return df


In [13]:
df = transform_data(data)

In [14]:
df

,datetime,open,high,low,close,volume
0,2026-02-04 15:37:00,277.670,277.785,277.570,277.700,9808
1,2026-02-04 15:36:00,277.330,277.740,277.330,277.650,11560
2,2026-02-04 15:35:00,277.280,277.510,277.000,277.340,12067
3,2026-02-04 15:34:00,277.450,277.620,277.220,277.295,11168
4,2026-02-04 15:33:00,277.345,277.555,277.340,277.460,3324
5,2026-02-04 15:32:00,277.060,277.500,277.010,277.325,4659
6,2026-02-04 15:31:00,277.350,277.360,277.030,277.040,3877
7,2026-02-04 15:30:00,277.660,277.730,277.330,277.330,4369
8,2026-02-04 15:29:00,278.020,278.080,277.570,277.610,8072
9,2026-02-04 15:28:00,277.940,277.990,277.725,277.990,6341


### Extract data for multiple symbols

In [15]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
all_data = []
def fetch_data(symbol, api_key):
    url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=5"
    response = requests.get(url)
    response.raise_for_status() #raise an error if we get a bad response
    data = response.json()

    if data.get('status') != 'ok':
        raise ValueError(f'Error fetching data for {symbol}: {data.get("message", "Unknown error")}')
    return data

In [46]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'META',]


for symbol in symbols:
    data = fetch_data(symbol, api_key)
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)

all_data


[             datetime    open     high      low    close  volume symbol
 0 2026-02-04 15:39:00  277.57  277.660  277.245  277.380   11410   AAPL
 1 2026-02-04 15:38:00  277.73  277.755  277.440  277.615    7362   AAPL
 2 2026-02-04 15:37:00  277.67  277.785  277.570  277.700    9808   AAPL
 3 2026-02-04 15:36:00  277.33  277.740  277.330  277.650   11560   AAPL
 4 2026-02-04 15:35:00  277.28  277.510  277.000  277.340   12067   AAPL,
              datetime     open     high      low   close  volume symbol
 0 2026-02-04 15:39:00  415.650  415.770  415.470  415.57    2463   MSFT
 1 2026-02-04 15:38:00  415.350  415.560  415.240  415.56    1904   MSFT
 2 2026-02-04 15:37:00  415.395  415.395  415.085  415.32    2877   MSFT
 3 2026-02-04 15:36:00  415.250  415.360  415.200  415.32    2613   MSFT
 4 2026-02-04 15:35:00  415.295  415.320  415.140  415.17    4236   MSFT,
              datetime    open     high      low   close  volume symbol
 0 2026-02-04 15:39:00  333.61  333.630  333.370  

### APEX WEALTH DATA PIPELINE


In [17]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [18]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [19]:

load_dotenv()
api_key = os.getenv('API_KEY')

In [20]:
API_KEY = os.getenv('API_KEY')

In [21]:
api_key

'0fb28af206c64e9da726186651ad43c2'

In [22]:
symbol = "AAPL"
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"


In [23]:
#Get response from the API
response = requests.get(url)
response.status_code

200

In [24]:
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "apikey": API_KEY
}


In [25]:


resp = requests.get(url, params=params)
print("Status code:", resp.status_code)  # Should be 200

data = resp.json()
print("Keys in JSON:", data.keys())      # Should include 'values'


Status code: 200
Keys in JSON: dict_keys(['meta', 'values', 'status'])


In [26]:
#get data in json format
data = response.json()
data


{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-02-04 15:38:00',
   'open': '277.73',
   'high': '277.755',
   'low': '277.44',
   'close': '277.615',
   'volume': '7362'},
  {'datetime': '2026-02-04 15:37:00',
   'open': '277.67',
   'high': '277.785',
   'low': '277.57',
   'close': '277.7',
   'volume': '9808'},
  {'datetime': '2026-02-04 15:36:00',
   'open': '277.33',
   'high': '277.74',
   'low': '277.33',
   'close': '277.65',
   'volume': '11560'},
  {'datetime': '2026-02-04 15:35:00',
   'open': '277.28',
   'high': '277.51',
   'low': '277',
   'close': '277.34',
   'volume': '12067'},
  {'datetime': '2026-02-04 15:34:00',
   'open': '277.45',
   'high': '277.62',
   'low': '277.22',
   'close': '277.295',
   'volume': '11168'},
  {'datetime': '2026-02-04 15:33:00',
   'open': '277.345',
   'high': '277.555'

In [27]:
df = pd.DataFrame(data["values"])
df.head()

,datetime,open,high,low,close,volume
0,2026-02-04 15:38:00,277.73,277.755,277.44,277.615,7362
1,2026-02-04 15:37:00,277.67,277.785,277.57,277.7,9808
2,2026-02-04 15:36:00,277.33,277.74,277.33,277.65,11560
3,2026-02-04 15:35:00,277.28,277.51,277,277.34,12067
4,2026-02-04 15:34:00,277.45,277.62,277.22,277.295,11168


In [28]:
#transform data to dataframe
def transform_data(data):
    #extract values
    time_series = data['values']

    #convert to dataframe
    df = pd.DataFrame(time_series)

    #convert to proper datatypes
    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int' })
    return df


In [29]:
df = transform_data(data)

In [30]:
df

,datetime,open,high,low,close,volume
0,2026-02-04 15:38:00,277.730,277.755,277.440,277.615,7362
1,2026-02-04 15:37:00,277.670,277.785,277.570,277.700,9808
2,2026-02-04 15:36:00,277.330,277.740,277.330,277.650,11560
3,2026-02-04 15:35:00,277.280,277.510,277.000,277.340,12067
4,2026-02-04 15:34:00,277.450,277.620,277.220,277.295,11168
5,2026-02-04 15:33:00,277.345,277.555,277.340,277.460,3324
6,2026-02-04 15:32:00,277.060,277.500,277.010,277.325,4659
7,2026-02-04 15:31:00,277.350,277.360,277.030,277.040,3877
8,2026-02-04 15:30:00,277.660,277.730,277.330,277.330,4369
9,2026-02-04 15:29:00,278.020,278.080,277.570,277.610,8072


### Extract data for multiple symbols

In [31]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'DIS']
all_data = []
def fetch_data(symbol, api_key):
    url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=5"
    response = requests.get(url)
    response.raise_for_status() #raise an error if we get a bad response
    data = response.json()

    if data.get('status') != 'ok':
        raise ValueError(f'Error fetching data for {symbol}: {data.get("message", "Unknown error")}')
    return data

In [32]:
#Symbols added to pipeline
# ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'DIS',]

In [33]:
# PR comment: Verified symbol list updated for diversification
# Symbols added for diversification in PR


In [49]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'DIS']

for symbol in symbols:
    data = fetch_data(symbol, api_key)
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)

all_data


ValueError: Error fetching data for AAPL: You have run out of API credits for the current minute. 11 API credits were used, with the current limit being 8. Wait for the next minute or consider switching to a higher tier plan at https://twelvedata.com/pricing

In [48]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'DIS',]

for symbol in symbols:
    data = fetch_data(symbol, api_key)
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)

all_data


ValueError: Error fetching data for AAPL: You have run out of API credits for the current minute. 10 API credits were used, with the current limit being 8. Wait for the next minute or consider switching to a higher tier plan at https://twelvedata.com/pricing

In [39]:
#concat - joining or stacking all the dataframes together
all_df = pd.concat(all_data, ignore_index=True)

### APEX WEALTH DATA PIPELINE


In [40]:
all_df

,datetime,open,high,low,close,volume,symbol
0,2026-02-04 15:39:00,277.570,277.660,277.245,277.380,11410,AAPL
1,2026-02-04 15:38:00,277.730,277.755,277.440,277.615,7362,AAPL
2,2026-02-04 15:37:00,277.670,277.785,277.570,277.700,9808,AAPL
3,2026-02-04 15:36:00,277.330,277.740,277.330,277.650,11560,AAPL
4,2026-02-04 15:35:00,277.280,277.510,277.000,277.340,12067,AAPL
5,2026-02-04 15:39:00,415.650,415.770,415.470,415.570,2463,MSFT
6,2026-02-04 15:38:00,415.350,415.560,415.240,415.560,1904,MSFT
7,2026-02-04 15:37:00,415.395,415.395,415.085,415.320,2877,MSFT
8,2026-02-04 15:36:00,415.250,415.360,415.200,415.320,2613,MSFT
9,2026-02-04 15:35:00,415.295,415.320,415.140,415.170,4236,MSFT


#### Loading

In [41]:
load_dotenv()
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')


In [42]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv(dotenv_path="C:/full/path/to/your/project/.env")  # change this to your actual path

# Get DB_NAME
DB_NAME = os.getenv("DB_NAME")
print("DB_NAME:", DB_NAME)


DB_NAME: apex_db


In [43]:
#create a database connnection url 

from sqlalchemy import create_engine 
import psycopg2


db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_url)

#load dataframe to postgres database
all_df.to_sql('stockPrices_data', engine, if_exists='append', index=False)

print("Data loaded to database successfully")             

Data loaded to database successfully


In [44]:
all_df.to_csv('stock_data.csv', index=False)

In [ ]:
#concat - joining or stacking all the dataframes together
all_df = pd.concat(all_data, ignore_index=True)

In [ ]:
all_df

,datetime,open,high,low,close,volume,symbol
0,2026-02-02 09:50:00,262.820,263.550,262.820,263.410,4157,AAPL
1,2026-02-02 09:49:00,262.440,262.830,262.275,262.830,6105,AAPL
2,2026-02-02 09:48:00,261.810,262.500,261.760,262.500,2840,AAPL
3,2026-02-02 09:47:00,261.895,261.955,261.520,261.680,4403,AAPL
4,2026-02-02 09:46:00,261.120,262.000,260.955,262.000,6754,AAPL
5,2026-02-02 09:50:00,428.670,429.225,428.600,429.090,13236,MSFT
6,2026-02-02 09:49:00,428.505,428.840,428.270,428.660,7789,MSFT
7,2026-02-02 09:48:00,427.865,428.505,427.690,428.505,4546,MSFT
8,2026-02-02 09:47:00,427.710,428.240,427.600,427.870,11550,MSFT
9,2026-02-02 09:46:00,427.950,428.130,427.700,427.745,4713,MSFT


#### Loading

In [ ]:
load_dotenv()
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')


In [41]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv(dotenv_path="C:/full/path/to/your/project/.env")  # change this to your actual path

# Get DB_NAME
DB_NAME = os.getenv("DB_NAME")
print("DB_NAME:", DB_NAME)


DB_NAME: apex_db


In [45]:
#create a database connnection url 

from sqlalchemy import create_engine 
import psycopg2


db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_url)

#load dataframe to postgres database
all_df.to_sql('stockPrices_data', engine, if_exists='append', index=False)

print("Data loaded to database successfully")             

Data loaded to database successfully


In [ ]:
all_df.to_csv('stock_data.csv', index=False)